# File Deletion (Purge) Job — Design

**System:** File Validation & Routing Framework (`fvr`, MongoDB + GridFS) • **Version:** 1.0 • **Status:** DRAFT FOR REVIEW

> **STORAGE ADDENDUM (2026-11-09):** GridFS is deprecated for portal uploads — bytes live in Databricks volumes (staging → intake/quarantine). Throughout this document, read "GridFS copy" as **volume copy**; `file_registry.gridfs_purged_at` is renamed **`volume_purged_at`** (LLD v1.3); byte deletion is a single `DELETE /api/2.0/fs/files{path}` per file. The gate logic (retention exceeded AND fresh MATCHED AND no legal hold) is unchanged.

## 1. Purpose & Scope

Deletes the **GridFS copy** of a file when it is provably safe to do so — **both** gates must pass:

1. **Retention exceeded** — the file is older than `retention_policies.purge_after_days` for its status class, **and**
2. **Reconciliation PASSED** — the latest `reconciliation_results` for the file is `MATCHED` (uploaded = routed = transferred = bronze-processed).

> **AND, not OR** (HLD decision D10): retention alone can delete a file Databricks never processed; a MATCHED result alone ignores the business retention window. Purging on `retention exceeded AND MATCHED` is the only safe combination. A legal-hold flag blocks purge absolutely regardless of both.

**In scope:** GridFS bytes (`fs.files` + `fs.chunks`) only. **Out of scope:** volume-side Intake/Quarantine copies (Databricks-side lifecycle — separate job), archive copies (`archive_after_days` cold storage — Phase 3a), and all metadata rows (**never deleted — tombstone pattern**).

**What is deleted vs kept**

| Object | On purge |
| --- | --- |
| `fs.files` + `fs.chunks` (the bytes) | **DELETED** (per-file, in a transaction) |
| `file_registry` doc | KEPT — gains `gridfs_purged_at` timestamp (see LLD additions) |
| `validation_results`, `file_transfers`, `audit_log` | KEPT — the compliance/evidence trail |
| `file_lifecycle` | NEW entry appended: `location: PURGED` |

## 2. Eligibility Gates (all must hold, per file)

| # | Gate | Source | Failure disposition |
| --- | --- | --- | --- |
| G1 | Policy scope match — file_type (or `*`) + `status_class` (INTAKE_COPY / QUARANTINED / UNIDENTIFIED) | `retention_policies` | skip (not in scope) |
| G2 | **Retention exceeded** — `received_at < now − purge_after_days` | `file_registry` | skip (too young) |
| G3 | **Reconciliation PASSED** — latest `reconciliation_results.status = MATCHED` within `reconciliation_lookback_days` (default 7) | `reconciliation_results` | skip + counter `no_match` (**escalate if stuck > 14 d** — see alerts) |
| G4 | Transfer verified — a COMPLETED `file_transfers` doc exists for the file (dest copy exists) | `file_transfers` | skip (no proof of copy) |
| G5 | No legal hold — `retention_policies.legal_hold = false` (plus optional per-file override, see LLD additions) | `retention_policies` | skip (hold) |
| G6 | Not already purged — `file_registry.gridfs_purged_at = null` and latest `file_lifecycle.location != PURGED` | `file_registry` / `file_lifecycle` | skip (idempotent re-run) |

**Query shape (candidate scan, index-driven):** `file_registry` by `{file_type_id, current_status ∈ {ROUTED_INTAKE, ROUTED_QUARANTINE}, received_at < cutoff, gridfs_purged_at: null}` → then G3/G4/G5 checked per candidate (≤ 2 lookups each, all indexed).

## 3. Job Algorithm

```text
run_purge_job(policy, mode):
  job ← archive_jobs.insert(policy_id, status=RUNNING, mode)          # even dry-runs are recorded
  cutoff ← now − policy.purge_after_days
  candidates ← file_registry.find(scope, received_at < cutoff, gridfs_purged_at null)
  for file in candidates (batches of 500):
      if NOT eligible(file, policy):  counters[reason]++; continue    # gates G3–G6
      if mode == DRY_RUN: report.append(file_id); continue
      with mongo_transaction():                                        # replica set txn
          fs.files.deleteOne({_id: file.gridfs_file_id})
          fs.chunks.deleteMany({files_id: file.gridfs_file_id})
          file_registry.updateOne({file_id, gridfs_purged_at: null},   # guarded update
                                  {$set: {gridfs_purged_at: now}})
          file_lifecycle.insertOne({location: PURGED, triggered_by_job: job})
          audit_log.insertOne({action: FILE_PURGED, actor: system:purge_job, ...})
  archive_jobs.updateOne(status=COMPLETED, counters)
  alert_if(counters.no_match_stuck > 0 || failures > 0)
```

**Guarded update** (`gridfs_purged_at: null` in the filter) makes the purge **idempotent**: a concurrent job or re-run can never double-delete or re-report — the update matches zero documents and the transaction aborts harmlessly.

### Transaction boundary

All four writes (bytes + tombstone + lifecycle + audit) happen in **one MongoDB transaction** on the replica set — either the file is fully purged with its evidence, or nothing changes. Partial purges cannot occur.

## 4. Safety Rails

* **Dry-run first (mandatory):** every new/edited retention policy runs one dry-run cycle — the report lists candidate `file_id`s + per-gate skip reasons before any deletion is enabled.
* **Batch limits:** max 500 files or 10 GB deleted per job run; leftover candidates roll to the next run (bounded blast radius).
* **Grace period:** `reconciliation_lookback_days` (default 7) — a MATCHED older than the lookback is re-verified by a fresh reconciliation run before purging; stale MATCHED verdicts don't authorize deletion.
* **Rate/quiet hours:** deletions throttled; job never runs during backup windows.
* **Approval:** enabling a purge policy (flipping `dry_run_only → false`) is an `ADMIN` config change with `approver` recorded in `audit_log`.
* **Tombstone pattern:** metadata rows are **never** deleted — "deleted file" = bytes gone + `gridfs_purged_at` set + `file_lifecycle[PURGED]` + `audit_log`. Compliance can always answer *what was deleted, when, why, and on whose authority*.
* **Reversibility:** bytes are gone permanently (that is the point) — but the Databricks volume copy + archive copy (if archived) still hold the data.

## 5. LLD Additions Required (this job)

| Change | Collection / field | Notes |
| --- | --- | --- |
| **NEW** `file_registry.gridfs_purged_at` | date, nullable, default null | eligibility query + tombstone; index `{file_type_id: 1, gridfs_purged_at: 1, current_status: 1}` (partial where null) for the candidate scan |
| **NEW** `retention_policies.reconciliation_lookback_days` | int, default 7 | G3 freshness window |
| **NEW** `retention_policies.dry_run_only` | bool, default true | first deploy of any policy never deletes |
| **NEW** `retention_policies.purge_requires_matched` | bool, default **true** | escape hatch for explicitly approved exceptions (e.g. quarantined junk that never reached bronze); flipping to false requires ADMIN + approver + audit |
| **NEW** `file_registry.legal_hold` (optional) | bool, nullable | per-file hold override on top of policy-level hold |
| **NEW** enum `AUDIT_ACTION` value `FILE_PURGED` | `audit_log.action` | matches `EVENT_TYPE.FILE_PURGED` |

## 6. Edge Cases & Dispositions

| Case | Disposition |
| --- | --- |
| File eligible by age, but never reconciled (`no MATCHED` ever) | **Never purged.** Counter `no_match` grows → alert at 14 d stuck → `MISSING_IN_BRONZE` investigation. This is the resilience promise in action. |
| Phase-1 legacy files (uploaded before reconciliation existed) | Held until the first reconciliation backfill run certifies them (`MATCHED` on a backfill `reconciliation_runs` doc) — or explicitly via `purge_requires_matched=false` with approval. |
| `MISSING_IN_BRONZE` / `ROW_MISMATCH` verdicts | Never purge; the discrepancy must be resolved (resolution_status → CLOSED) and a fresh `MATCHED` obtained. |
| `STUCK_IN_TRANSIT` (transfer never COMPLETED) | G4 blocks purge; transfer retry loop + alerting owns it. |
| Quarantined files (`status_class: QUARANTINED`) | Same gates apply — quarantined ≠ deleted; they get their own `purge_after_days` (typically shorter) but still need a MATCHED-style verdict where one exists, else the explicit-approval escape hatch. |
| Duplicate `event_log` re-delivery while purge runs | Irrelevant — idempotency is keyed on `file_registry`, not events. |
| Job crash mid-batch | Transactions guarantee per-file atomicity; already-purged files are skipped via G6 on re-run. |

## 7. Pseudocode (Python, pymongo/motor)

```python
# PSEUDOCODE — purge job worker (backend/app/services/archival.py)
def eligible(file, policy, recon_lookback):
    # G3 — reconciliation PASSED (fresh)
    latest = db.reconciliation_results.find_one(
        {"file_id": file["file_id"],
         "status": "MATCHED",
         "executed_at": {"$gte": utcnow() - recon_lookback}},
        sort=[("executed_at", -1)])
    if not latest:
        return False, "no_match"
    # G4 — verified transfer copy exists
    t = db.file_transfers.find_one(
        {"file_id": file["file_id"], "status": "COMPLETED"})
    if not t:
        return False, "no_transfer"
    # G5 — no legal hold
    if policy["legal_hold"] or file.get("legal_hold"):
        return False, "legal_hold"
    return True, None

def run_purge_job(policy_id, mode="DRY_RUN"):
    policy = db.retention_policies.find_one({"policy_id": policy_id})
    if mode == "LIVE" and policy["dry_run_only"]:
        raise PolicyRequiresDryRun
    job_id = db.archive_jobs.insert_one({...})["job_id"]
    cutoff = utcnow() - timedelta(days=policy["purge_after_days"])
    counters = Counter(); report = []
    for file in scan_candidates(policy, cutoff, batch=500):
        ok, reason = eligible(file, policy, lookback(policy))
        if not ok:
            counters[reason] += 1; continue
        if mode == "DRY_RUN":
            report.append(file["file_id"]); continue
        with client.start_session() as s, s.start_transaction():
            gridfs_delete(s, file["gridfs_file_id"])                 # fs.files + fs.chunks
            db.file_registry.update_one(
                {"file_id": file["file_id"], "gridfs_purged_at": None},
                {"$set": {"gridfs_purged_at": utcnow()}}, session=s)  # guarded
            db.file_lifecycle.insert_one({..., "location": "PURGED"}, session=s)
            db.audit_log.insert_one({"action": "FILE_PURGED", ...}, session=s)
        counters["purged"] += 1
    finalize_job(job_id, counters, report)
    alert_stuck_no_match(counters, policy)
```

**Schedule:** daily off-peak (e.g. 02:00), one run per active retention policy; job control + counters in `archive_jobs`; alerts via `notifications` (`event: FILE_PURGED` stuck/failure conditions).

## 8. Review Checklist

1. **Gate logic** — confirm `AND` semantics (retention **AND** MATCHED) and the `purge_requires_matched=false` escape hatch policy.
2. **Retention numbers** — `purge_after_days` per status class (INTAKE_COPY vs QUARANTINED vs UNIDENTIFIED) and the lookback window (7 d proposed).
3. **Tombstone retention** — metadata rows kept forever (compliance) or do you eventually archive them too?
4. **Legal hold granularity** — policy-level only, or the per-file `file_registry.legal_hold` override as proposed?
5. **Legacy Phase-1 files** — hold for reconciliation backfill, or approved early purge via the escape hatch?
6. **Volume-side copies** — the Databricks Intake/Quarantine copies need their own lifecycle job (out of scope here) — confirm the target date.

---
*Companion docs: `Database Design HLD` (D10 purge gate) • `Database Design LLD` (retention_policies, archive_jobs, file_lifecycle) • `ER Diagram` (archive chain: retention_policies → archive_jobs → file_lifecycle).*